# NZZ ContextAI — Pipeline Runner

In [1]:
import sys, importlib
sys.path.insert(0, "../src")
sys.path.insert(0, "../scripts")

import config, ingest, experiment, evaluate
importlib.reload(config)
importlib.reload(ingest)
importlib.reload(experiment)
importlib.reload(evaluate)

print("Imports OK")

Imports OK


## 1 — Indexierung

In [2]:
ingest.ingest(month="2025_12", reset=True)

Lade Monat: 2025_12
Artikel: 1,763 geladen  →  1,746 nach Preprocessing
Chunks:  3,142
ChromaDB Collection geleert.
Lädt embedding model: Qwen/Qwen3-Embedding-0.6B


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Generiere Embeddings für 3142 Chunks...


Batches:   0%|          | 0/99 [00:00<?, ?it/s]

Speichere 3142 Chunks in ChromaDB...
Fertig — 3142 Chunks gespeichert.

✓ Indexierung abgeschlossen — 3,142 Chunks in ChromaDB


## 1 — Referenzantworten generieren

In [3]:
import build_expected_answers
importlib.reload(build_expected_answers)

build_expected_answers.main()

Lade Artikel...
[1/14] Bereits vorhanden — übersprungen
[2/14] Bereits vorhanden — übersprungen
[3/14] Bereits vorhanden — übersprungen
[4/14] Bereits vorhanden — übersprungen
[5/14] Bereits vorhanden — übersprungen
[6/14] Bereits vorhanden — übersprungen
[7/14] Bereits vorhanden — übersprungen
[8/14] Bereits vorhanden — übersprungen
[9/14] Bereits vorhanden — übersprungen
[10/14] Bereits vorhanden — übersprungen
[11/14] Bereits vorhanden — übersprungen
[12/14] Bereits vorhanden — übersprungen
[13/14] Bereits vorhanden — übersprungen
[14/14] Bereits vorhanden — übersprungen

✓ 0 Referenzantworten generiert und in ground_truth.jsonl gespeichert.


## 2 — Ground Truth prüfen

In [4]:
import json
from config import EVAL_GROUND_TRUTH

with open(EVAL_GROUND_TRUTH, encoding="utf-8") as f:
    samples = [json.loads(line) for line in f if line.strip()]

complete = [s for s in samples if len(s.get("expected_answer") or "") > 80]
print(f"Ground-Truth Einträge: {len(samples)}")
print(f"Mit Referenzantwort:   {len(complete)} / {len(samples)}")
print()
for s in samples:
    answer = s.get("expected_answer") or ""
    status = "✓" if len(answer) > 80 else "✗"
    print(f"  {status}  {s['query'][:65]}")

Ground-Truth Einträge: 14
Mit Referenzantwort:   14 / 14

  ✓  Was erwarten Schweizer Anlageprofis für die Börsenmärkte im Jahr 
  ✓  Werden in der Schweiz wieder Negativzinsen eingeführt?
  ✓  Wie verhält sich Putins Kriegsführung im Vergleich zu Stalin?
  ✓  Was zeigen sowjetische Archivdokumente über die Denkweise russisc
  ✓  Wie werden ukrainische Kinder in Russland umerzogen?
  ✓  Russland verschleppt ukrainische Kinder und gibt sie zur Adoption
  ✓  Ukrainischer Freiwilliger birgt tote russische Soldaten vom Schla
  ✓  Menschlichkeit im Krieg: Umgang mit gefallenen Feinden im Donbass
  ✓  Wie hat sich die Schweiz für die Zurückweisung jüdischer Flüchtli
  ✓  Kaspar Villiger Entschuldigung Bundesrat Holocaust 1995
  ✓  Warum kann man Grundstücke besitzen und wie entstand das Eigentum
  ✓  Geschichte des Landbesitzes Schweiz Immobilienpreise Eigentumsrec
  ✓  Wie beeinflusst künstliche Intelligenz das menschliche Denken und
  ✓  Menschliche Dummheit und Kognitionsforschung im Zeit

## 3 — Vollständige Pipeline + MLflow

In [8]:
from experiment import run_experiment

run_experiment()

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


KeyboardInterrupt: 

## 4 — MLflow UI starten

In [ ]:
import subprocess, sys

proc = subprocess.Popen(
    [sys.executable, "-m", "mlflow", "ui",
     "--backend-store-uri", "sqlite:///../mlflow.db",
     "--port", "5000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("MLflow UI läuft → http://localhost:5000")
print(f"(Prozess-ID: {proc.pid} — Kernel neu starten zum Beenden)")

## 5 — Einzelne Query testen

In [6]:
from embed import get_chroma_collection
from retrieval import load_models, retrieve
from generate import generate
from config import (
    CHROMA_PATH, CHROMA_COLLECTION,
    USE_RERANKING, EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
)

collection      = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
model, reranker = load_models(use_reranking=USE_RERANKING)
print(f"Bereit — {collection.count():,} Chunks in ChromaDB")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Bereit — 3,142 Chunks in ChromaDB


In [7]:
QUERY = "Was berichtete die NZZ über die SNB?"

chunks = retrieve(QUERY, model, collection, reranker,
                  top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
                  top_k_rerank=EVAL_TOP_K_FINAL)

print(f"Top-{len(chunks)} Quellen:\n")
for i, c in enumerate(chunks, 1):
    score = c.get("rerank_score", c.get("similarity_score", 0))
    print(f"  [{i}] {c['title'][:65]}  (score: {score:.3f})")

print("\nAntwort:\n")
print(generate(QUERY, chunks))

Top-5 Quellen:

  [1] Briefing am Donnerstagabend  (score: 2.914)
  [2] Der NZZ-Adventskalender blickt zurück: der Anfang vom Ende der Cr  (score: -0.185)
  [3] Die SNB bleibt beim Nullzins und widerspricht der UBS  (score: -0.313)
  [4] Briefing am Donnerstagabend  (score: -0.896)
  [5] Schweiz: SVP-Nationalrat David Zuberbühler gibt seinen Rücktritt   (score: -1.523)

Antwort:

Die NZZ berichtete, dass die Schweizerische Nationalbank (SNB) ihren Leitzins unverändert bei 0,0 Prozent belassen hat, was die Erwartungen der meisten Beobachter bestätigte. Die SNB hält an dieser Nullzinspolitik fest, da sie davon ausgeht, dass diese Massnahme die Wirtschaft stärkt und die Preisstabilität sichert. Darüber hinaus unterstützt die SNB die geplanten Verschärfungen der Eigenmittelvorschriften für Banken und betrachtet diese als zumutbar. Die Währungshüter prognostizieren für das laufende Jahr einen durchschnittlichen Preisanstieg von 0,2 Prozent und für das kommende Jahr 0,3 Prozent. Die SNB beto